# Noise floor from the 2022 and 2023 epoch pair

Runs the LiDAR step that needs PDAL. Everything else in the project runs in
the working environment on a laptop.

The two acquisitions are one year apart at comparable return density. One
year of true canopy growth is small against the expected measurement error,
so the spread of their difference over stable cells estimates that error
directly. The result gates whether span-scale change is detectable at all.

Point clouds are read by bounding box over HTTPS. No point-cloud file is
downloaded.

In [ ]:
!pip -q install condacolab
import condacolab
condacolab.install()

The kernel restarts after the cell above. Continue from the next cell.

In [ ]:
REPO = "https://github.com/Lakshaycodes08/CanopyGuard-AI.git"
BRANCH = "phase1/reconciliation-and-pipelines"
TILES = 40

!git clone --branch $BRANCH --depth 1 $REPO /content/CanopyGuard-AI
%cd /content/CanopyGuard-AI
!mamba env update -n base -f environment-lidar.yml -q
!pip -q install -e . --no-deps

In [ ]:
import pdal
import rasterio

from canopyguard.config import load_config
from canopyguard.lidar.plan import calibration_plan

config = load_config("configs/lidar.yaml")
plan = calibration_plan(config, TILES)
config["harmonization"]["sample_radius_m"] = plan["sample_radius_m"]

print(f"pdal {pdal.__version__}")
print(f"{plan['tile_count']} tiles, {plan['sample_area_km2']:.1f} km2")
print(f"sampling radius {plan['sample_radius_m']:.4f} m")
print(f"projects {plan['projects']}")

## One tile first

Confirms the resource is reachable and the pipeline runs before committing to
the full sample.

In [ ]:
from pathlib import Path

from canopyguard.lidar.chm import build_terrain_pipeline, run_pipeline

OUT = Path("/content/truth")
OUT.mkdir(parents=True, exist_ok=True)

probe = plan["tiles"][0]
for epoch, reader in sorted(probe["readers"].items()):
    stem = f"{epoch}_t{probe['index']:04d}"
    pipeline = build_terrain_pipeline(
        reader, str(OUT / f"dtm_{stem}.tif"), str(OUT / f"dsm_{stem}.tif"), config
    )
    print(epoch, "points:", run_pipeline(pipeline))

## Full sample

A tile that returns no points is recorded and skipped. The published
acquisition boundary is a hull, so gaps inside it are expected.

In [ ]:
import json

results = []
for tile in plan["tiles"]:
    for epoch, reader in sorted(tile["readers"].items()):
        stem = f"{epoch}_t{tile['index']:04d}"
        pipeline = build_terrain_pipeline(
            reader, str(OUT / f"dtm_{stem}.tif"), str(OUT / f"dsm_{stem}.tif"), config
        )
        try:
            count = run_pipeline(pipeline)
        except RuntimeError as error:
            count, note = 0, str(error)[:200]
        else:
            note = ""
        results.append({"tile": tile["index"], "epoch": epoch, "points": count, "note": note})
        print(stem, count, note)

Path(OUT / "build_log.json").write_text(json.dumps(results, indent=2))

## Canopy height, co-registration, noise floor

The canopy height model is the surface minus the terrain within one epoch, so
the vertical datum cancels. 2023 is co-registered onto 2022 on the bare-earth
surface before the two are differenced.

In [ ]:
import numpy as np

from canopyguard.evaluation.detectability import noise_floor_table
from canopyguard.lidar.coreg import accept, align, apply_shift
from canopyguard.lidar.difference import difference_ladder
from canopyguard.lidar.gridio import read_grid, write_grid

settings = config["coregistration"]
resolution = config["chm"]["resolution_m"]
scales = config["aggregation"]["scales_m"]
min_valid = config["aggregation"]["min_valid_fraction"]

pooled = {float(scale): [] for scale in scales}
shifts = []

for tile in plan["tiles"]:
    stem_a = f"2022_t{tile['index']:04d}"
    stem_b = f"2023_t{tile['index']:04d}"
    paths = [OUT / f"{kind}_{stem}.tif" for stem in (stem_a, stem_b) for kind in ("dtm", "dsm")]
    if not all(path.exists() for path in paths):
        continue

    dtm_a, profile = read_grid(OUT / f"dtm_{stem_a}.tif")
    dsm_a, _ = read_grid(OUT / f"dsm_{stem_a}.tif")
    dtm_b, _ = read_grid(OUT / f"dtm_{stem_b}.tif")
    dsm_b, _ = read_grid(OUT / f"dsm_{stem_b}.tif")
    if dtm_a.shape != dtm_b.shape:
        continue

    shift = align(dtm_a, dtm_b, resolution, settings["min_slope_deg"], settings["max_slope_deg"],
                  settings["max_iterations"], settings["convergence_tolerance_m"])
    shift["tile"] = tile["index"]
    shift["accepted"] = float(accept(shift, settings["max_accepted_shift_m"]))
    shifts.append(shift)
    if not shift["accepted"]:
        continue

    chm_a = np.clip(dsm_a - dtm_a, config["chm"]["clamp_min_m"], config["chm"]["clamp_max_m"])
    chm_b = np.clip(dsm_b - dtm_b, config["chm"]["clamp_min_m"], config["chm"]["clamp_max_m"])
    chm_b = apply_shift(chm_b, shift, resolution)
    write_grid(OUT / f"chm_{stem_a}.tif", chm_a, profile)
    write_grid(OUT / f"chm_{stem_b}.tif", chm_b, profile)

    for scale, delta in difference_ladder(chm_a, chm_b, resolution, scales, min_valid).items():
        pooled[scale].append(np.asarray(delta).ravel())

ladder = {scale: np.concatenate(parts) for scale, parts in pooled.items() if parts}
rows = noise_floor_table(ladder, base_scale_m=resolution)
Path(OUT / "noise_floor.json").write_text(json.dumps({"rows": rows, "shifts": shifts}, indent=2))
for row in rows:
    print(f"{row['scale_m']:>7.0f} m  sigma {row['sigma_m']:6.3f}  lod95 {row['lod95_m']:6.3f}  mean {row['mean_m']:+6.3f}  n {row['cells']:,.0f}")
print(f"decay exponent {rows[0]['decay_exponent']:.3f}")

## Gate

Proceed to the study-area work only if every condition holds.

In [ ]:
by_scale = {row["scale_m"]: row for row in rows}
accepted = [s for s in shifts if s["accepted"]]
median_shift = float(np.median([s["magnitude_m"] for s in accepted])) if accepted else float("nan")
sigmas = [row["sigma_m"] for row in rows]

checks = {
    "mean one-year change in [0.0, 1.5] m": 0.0 <= by_scale[min(by_scale)]["mean_m"] <= 1.5,
    "sigma falls with scale": all(b <= a for a, b in zip(sigmas, sigmas[1:])),
    "median tile shift under 1.5 m": median_shift < 1.5,
    "sigma at 100 m under 2.0 m": by_scale.get(100.0, {}).get("sigma_m", 9e9) < 2.0,
}
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")
print("\nGATE", "PASS" if all(checks.values()) else "FAIL")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/CanopyGuard/truth"
!cp /content/truth/*.json "/content/drive/MyDrive/CanopyGuard/truth/"
!cp /content/truth/chm_*.tif "/content/drive/MyDrive/CanopyGuard/truth/"
!du -sh "/content/drive/MyDrive/CanopyGuard/truth"